In [1]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# Generate 120,000 transaction rows
n_rows = 120000

# 1. Customer IDs (5,000 unique customers)
customer_ids = np.random.randint(10001, 15001, size=n_rows)

# 2. Invoices (~25,000 unique invoice numbers)
invoice_numbers = np.random.randint(500000, 525000, size=n_rows)

# 3. Dates spanning 1 full year (2025-01-01 to 2025-12-31)
start_date = pd.to_datetime("2025-01-01")
end_date = pd.to_datetime("2025-12-31")
random_days = np.random.randint(0, (end_date - start_date).days, size=n_rows)
random_seconds = np.random.randint(0, 86400, size=n_rows)
invoice_dates = start_date + pd.to_timedelta(
    random_days, unit="D"
) + pd.to_timedelta(random_seconds, unit="s")

# 4. Quantity & Unit Price
quantities = np.random.randint(1, 15, size=n_rows)
unit_prices = np.round(np.random.uniform(2.5, 150.0, size=n_rows), 2)

# 5. Countries
countries = np.random.choice(
    ["United Kingdom", "Germany", "France", "EIRE", "Spain", "Netherlands"],
    size=n_rows,
    p=[0.75, 0.08, 0.07, 0.04, 0.03, 0.03],
)

# Combine into a DataFrame
df = pd.DataFrame(
    {
        "InvoiceNo": invoice_numbers,
        "CustomerID": customer_ids,
        "InvoiceDate": invoice_dates,
        "Quantity": quantities,
        "UnitPrice": unit_prices,
        "Country": countries,
    }
)

# Calculate total spending per transaction line
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

# Preview the first 5 rows and dataset details
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (120000, 7)


,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,Country,TotalAmount
0,505682,10861,2025-09-04 05:40:41,1,77.80,United Kingdom,77.80
1,521326,13773,2025-08-26 05:46:27,13,119.78,Germany,1557.14
2,504999,13093,2025-10-15 07:06:28,2,86.83,United Kingdom,173.66
3,501413,10467,2025-07-02 04:38:00,14,46.00,United Kingdom,644.00
4,521621,14427,2025-07-09 04:25:15,5,146.92,United Kingdom,734.60


In [2]:
import datetime as dt

# 1. Define snapshot date (1 day after the latest transaction date)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(f"Snapshot Date for Recency calculation: {snapshot_date}")

# 2. Aggregate metrics by CustomerID
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'TotalAmount': 'sum'                                      # Monetary
}).reset_index()

# 3. Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 4. Round Monetary value to 2 decimal places
rfm['Monetary'] = rfm['Monetary'].round(2)

# Check summary stats and preview top 5 rows
print("\n--- RFM Summary Statistics ---")
print(rfm.describe())

print("\n--- First 5 Customer RFM Profiles ---")
rfm.head()

Snapshot Date for Recency calculation: 2025-12-31 23:58:36

--- RFM Summary Statistics ---
         CustomerID      Recency    Frequency      Monetary
count   5000.000000  5000.000000  5000.000000   5000.000000
mean   12500.500000    15.800000    23.991200  13738.914356
std     1443.520003    14.983074     4.860643   3665.667411
min    10001.000000     1.000000    10.000000   3125.070000
25%    11250.750000     5.000000    21.000000  11111.700000
50%    12500.500000    11.000000    24.000000  13524.900000
75%    13750.250000    22.000000    27.000000  16076.210000
max    15000.000000   141.000000    46.000000  32080.500000

--- First 5 Customer RFM Profiles ---


,CustomerID,Recency,Frequency,Monetary
0,10001,10,25,15861.06
1,10002,14,25,18548.11
2,10003,46,17,5636.63
3,10004,18,29,12723.60
4,10005,28,19,9689.71


In [3]:
# 1. Recency Score (Inverted: lower days = better score)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# 2. Frequency Score (Higher count = better score; rank method handles ties)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# 3. Monetary Score (Higher spend = better score)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# 4. Concatenate scores into RFM_Score string and calculate total sum
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Preview customer scores
print("--- RFM Scores Generated Successfully ---")
rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Total']].head()

--- RFM Scores Generated Successfully ---


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,10001,10,25,15861.06,3,3,4,334,10
1,10002,14,25,18548.11,3,3,5,335,11
2,10003,46,17,5636.63,1,1,1,111,3
3,10004,18,29,12723.60,2,5,3,253,10
4,10005,28,19,9689.71,1,1,1,111,3


In [4]:
import re

# Define segmentation logic based on R and F scores
def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    
    if r in [4, 5] and f in [4, 5]:
        return 'Champions'
    elif r in [3, 4, 5] and f in [3, 4, 5]:
        return 'Loyal Customers'
    elif r in [4, 5] and f in [1, 2]:
        return 'Recent / New Customers'
    elif r in [3, 4] and f in [1, 2]:
        return 'Promising'
    elif r == 3 and f == 3:
        return 'Needs Attention'
    elif r in [1, 2] and f in [3, 4, 5]:
        return 'At Risk'
    elif r == 1 and f == 5:
        return 'Cant Lose Them'
    elif r in [1, 2] and f in [1, 2]:
        return 'Hibernating / Lost'
    else:
        return 'About to Sleep'

# Apply segmentation
rfm['Customer_Segment'] = rfm.apply(segment_customer, axis=1)

# Summary of customer count and total revenue per segment
segment_summary = rfm.groupby('Customer_Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Total_Revenue=('Monetary', 'sum'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean')
).reset_index().sort_values(by='Customer_Count', ascending=False)

segment_summary['Revenue_Share_%'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100).round(2)

print("--- Segment Distribution Summary ---")
segment_summary

--- Segment Distribution Summary ---


,Customer_Segment,Customer_Count,Total_Revenue,Avg_Recency,Avg_Frequency,Revenue_Share_%
3,Loyal Customers,1036,15220591.26,8.752896,25.674710,22.16
1,Champions,1028,17093352.73,4.468872,28.892023,24.88
2,Hibernating / Lost,949,10342510.47,32.584826,19.109589,15.06
0,At Risk,936,14353662.59,28.590812,26.705128,20.89
5,Recent / New Customers,699,7757286.72,4.685265,19.526466,11.29
4,Promising,352,3927168.01,12.440341,19.534091,5.72


In [5]:
from sqlalchemy import create_engine

# 1. Export CSV Files for Direct Power BI Import
df.to_csv("cleaned_transactions.csv", index=False)
rfm.to_csv("rfm_segmented_customers.csv", index=False)
segment_summary.to_csv("rfm_segment_summary.csv", index=False)

print("✅ CSV files saved successfully to your local directory!")

# 2. Push Tables to PostgreSQL
DB_USER = "postgres"
DB_PASSWORD = "amlan123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "ecommerce_rfm"

# Create SQLAlchemy connection engine
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

try:
    # Write transactions and segmented RFM metrics into PostgreSQL tables
    df.to_sql(
        "fact_transactions",
        engine,
        if_exists="replace",
        index=False,
        chunksize=10000,
    )
    rfm.to_sql(
        "dim_customer_rfm", engine, if_exists="replace", index=False
    )
    segment_summary.to_sql(
        "agg_rfm_summary", engine, if_exists="replace", index=False
    )

    print(
        "✅ Data successfully exported to PostgreSQL tables: 'fact_transactions', 'dim_customer_rfm', and 'agg_rfm_summary'!"
    )
except Exception as e:
    print(f"⚠️ Could not connect to PostgreSQL. Error details:\n{e}")
    print(
        "💡 Note: Make sure you have created the database 'ecommerce_rfm' in pgAdmin/PostgreSQL beforehand."
    )

✅ CSV files saved successfully to your local directory!
✅ Data successfully exported to PostgreSQL tables: 'fact_transactions', 'dim_customer_rfm', and 'agg_rfm_summary'!
